# Stage 2 — Supervised Fine-Tuning (SFT)

**Goal:** Teach the CPT-aligned model to translate English→Swahili and follow Swahili instructions.

## What changed from Stage 1
- We now know real step time is **~35s/step** (gradient checkpointing compute cost)
- Safe budget: `39,600s available ÷ 35s/step = ~1,100 steps`
- Dataset sized to exactly fit: `1,100 × 16 batch = ~17,600 samples`
- We load the **checkpoint-1000** adapter from Stage 1 instead of the bare base model

## Dataset mix — 40% parallel translation : 60% instruction
| Dataset | Type | Samples | License |
|---------|------|---------|--------|
| SAWA Corpus | Parallel EN→SW | 5,000 | CC-BY |
| Sunbird/SALT | Parallel EN→SW | 2,000 | Open |
| Aya Dataset (human) | Instruction SW | 4,000 | Apache 2.0 |
| Alpaca Swahili | Instruction SW | 4,000 | CC-BY-NC-4.0 |
| KenSwQuAD | QA SW | 2,000 | CC-BY-4.0 |

**Total: ~17,000 samples → ~1,062 steps**

## Key constraint
Bilingual input (EN or SW accepted), **Swahili-only output** — this is what gives the model its parameter efficiency advantage.

> **Kaggle setup:** GPU T4 x2 · Internet ON · HF_TOKEN secret set  
> **Add Data:** Your Work → stage1_cpt → adds checkpoint-1000 at `/kaggle/input/stage1-cpt/`

## 1. Lock GPU and install dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
!pip install -q transformers accelerate datasets peft trl bitsandbytes sacrebleu huggingface_hub

## 2. Environment check

In [ ]:
import torch
import transformers, peft, trl, platform

print("=" * 55)
print("ENVIRONMENT")
print("=" * 55)
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU 0        : {torch.cuda.get_device_name(0)} ({mem:.1f} GB)")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Active device: {DEVICE}")

## 3. Authenticate

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace auth successful.")
except Exception as e:
    print(f"Auth failed: {e}")

## 4. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
MODEL_ID        = "google/gemma-3-1b-it"
OUTPUT_DIR      = "/kaggle/working/stage2_sft_adapter"

# Stage 1 checkpoint — mounted via Add Data → stage1_cpt
# Try both possible mount paths
CPT_ADAPTER_PATH = None
for candidate in [
    "/kaggle/input/stage1-cpt/stage1_cpt_adapter/checkpoint-1000",
    "/kaggle/input/stage1cpt/stage1_cpt_adapter/checkpoint-1000",
    "/kaggle/input/stage1_cpt/stage1_cpt_adapter/checkpoint-1000",
]:
    if os.path.exists(candidate):
        CPT_ADAPTER_PATH = candidate
        break

if CPT_ADAPTER_PATH:
    print(f"CPT adapter found      : {CPT_ADAPTER_PATH}")
else:
    print("WARNING: CPT adapter not found — will train from base model.")
    print("Add stage1_cpt output as a dataset via Add Data → Your Work.")

# ── Data — sized for ~1,100 steps at batch 16 ─────────────────
SAWA_SAMPLES    = 5_000
SALT_SAMPLES    = 2_000
AYA_SAMPLES     = 4_000
ALPACA_SAMPLES  = 4_000
KENSWQUAD_SAMPLES = 2_000

MAX_SEQ_LEN     = 512    # shorter than CPT — translation pairs are shorter than forum posts

# ── Training ──────────────────────────────────────────────────
LEARNING_RATE   = 2e-4   # higher than CPT — SFT needs faster task adaptation
LR_SCHEDULER    = "cosine"
NUM_EPOCHS      = 1
BATCH_SIZE      = 2
GRAD_ACCUM      = 8      # effective batch = 16
WARMUP_STEPS    = 50
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
SAVE_STEPS      = 300
LOGGING_STEPS   = 25

# ── LoRA — attention only for SFT (MLP already aligned by CPT) ─
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_samples = SAWA_SAMPLES + SALT_SAMPLES + AYA_SAMPLES + ALPACA_SAMPLES + KENSWQUAD_SAMPLES
est_steps     = int(total_samples * 0.98) // (BATCH_SIZE * GRAD_ACCUM)
est_hours     = est_steps * 35 / 3600

print(f"Total samples          : {total_samples:,}")
print(f"Estimated steps        : {est_steps:,}")
print(f"Estimated time         : {est_hours:.1f} hours @ 35s/step")
print(f"Max sequence length    : {MAX_SEQ_LEN}")
print(f"LoRA targets           : {LORA_TARGET_MODULES}")

## 5. Load tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size:,}")

## 6. Prompt builder

All examples — translation pairs, instruction QA, reading comprehension — are wrapped in the same bilingual prompt template. Output is always Swahili only.

In [ ]:
def make_translation_prompt(english: str, swahili: str) -> str:
    """For parallel EN→SW pairs."""
    return (
        f"<start_of_turn>user\n"
        f"Translate the following English sentence to Swahili.\n"
        f"English: {english.strip()}\n"
        f"Swahili:<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{swahili.strip()}<end_of_turn>"
    )

def make_instruction_prompt(instruction: str, response: str, input_text: str = "") -> str:
    """For instruction-following and QA pairs."""
    user_part = instruction.strip()
    if input_text and input_text.strip():
        user_part += f"\n{input_text.strip()}"
    return (
        f"<start_of_turn>user\n"
        f"{user_part}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{response.strip()}<end_of_turn>"
    )

# Sanity check
sample = make_translation_prompt(
    "The children are playing in the park.",
    "Watoto wanacheza bustanini."
)
print("Sample translation prompt:")
print(sample)
print(f"\nTokens: {len(tokenizer.encode(sample))}")

## 7. Load and format datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

all_prompts = []

# ── 1. SAWA Corpus (parallel EN→SW) ───────────────────────────
print("Loading SAWA corpus...")
try:
    sawa = load_dataset("Helsinki-NLP/opus-100", "en-sw", split=f"train[:{SAWA_SAMPLES}]")
    for row in sawa:
        en = row.get("translation", {}).get("en", "")
        sw = row.get("translation", {}).get("sw", "")
        if en and sw and len(en) > 10 and len(sw) > 10:
            all_prompts.append(make_translation_prompt(en, sw))
    print(f"  SAWA loaded          : {len(all_prompts):,} pairs")
except Exception as e:
    print(f"  SAWA failed ({e}), trying alternative...")
    try:
        sawa = load_dataset("Nguyen-Thi-Thanh/sawa-en-sw", split=f"train[:{SAWA_SAMPLES}]")
        for row in sawa:
            en = row.get("en", row.get("english", ""))
            sw = row.get("sw", row.get("swahili", ""))
            if en and sw:
                all_prompts.append(make_translation_prompt(en, sw))
        print(f"  SAWA alt loaded      : {len(all_prompts):,} pairs")
    except Exception as e2:
        print(f"  SAWA alt also failed ({e2}) — skipping")

In [ ]:
# ── 2. Sunbird SALT (parallel EN→SW) ──────────────────────────
print("Loading Sunbird SALT...")
salt_before = len(all_prompts)
try:
    salt = load_dataset("Sunbird/salt", split=f"train[:{SALT_SAMPLES}]")
    print(f"  SALT columns         : {salt.column_names}")
    print(f"  SALT sample          : {dict(list(salt[0].items())[:4])}")
    for row in salt:
        # SALT has English + target language columns
        en = row.get("English", row.get("english", row.get("en", "")))
        sw = row.get("Swahili", row.get("swahili", row.get("sw", "")))
        if en and sw and len(str(en)) > 5 and len(str(sw)) > 5:
            all_prompts.append(make_translation_prompt(str(en), str(sw)))
    print(f"  SALT added           : {len(all_prompts) - salt_before:,} pairs")
except Exception as e:
    print(f"  SALT failed ({e}) — skipping")

In [ ]:
# ── 3. Aya Dataset — human-annotated Swahili instructions ──────
print("Loading Aya Dataset (human-annotated Swahili)...")
aya_before = len(all_prompts)
try:
    aya = load_dataset(
        "CohereForAI/aya_dataset",
        split="train",
        streaming=False
    )
    # Filter Swahili rows only
    aya_sw = aya.filter(lambda x: x.get("language", "") in ["swahili", "sw", "Swahili"])
    aya_sw = aya_sw.select(range(min(AYA_SAMPLES, len(aya_sw))))
    for row in aya_sw:
        inp  = row.get("inputs", row.get("input", row.get("prompt", "")))
        out  = row.get("targets", row.get("target", row.get("completion", "")))
        if inp and out:
            all_prompts.append(make_instruction_prompt(inp, out))
    print(f"  Aya SW rows added    : {len(all_prompts) - aya_before:,}")
except Exception as e:
    print(f"  Aya failed ({e}) — skipping")

In [ ]:
# ── 4. Alpaca Swahili ──────────────────────────────────────────
print("Loading Alpaca Swahili...")
alpaca_before = len(all_prompts)
try:
    alpaca = load_dataset(
        "Eshwardg2003/alpaca-swahili",
        split=f"train[:{ALPACA_SAMPLES}]"
    )
    print(f"  Alpaca columns       : {alpaca.column_names}")
    for row in alpaca:
        instruction = row.get("instruction", "")
        inp         = row.get("input", "")
        out         = row.get("output", "")
        if instruction and out:
            all_prompts.append(make_instruction_prompt(instruction, out, inp))
    print(f"  Alpaca added         : {len(all_prompts) - alpaca_before:,}")
except Exception as e:
    print(f"  Alpaca primary failed ({e}), trying alternate...")
    try:
        alpaca = load_dataset(
            "Emmanuel-Kwame/swahili-alpaca",
            split=f"train[:{ALPACA_SAMPLES}]"
        )
        for row in alpaca:
            instruction = row.get("instruction", "")
            inp         = row.get("input", "")
            out         = row.get("output", "")
            if instruction and out:
                all_prompts.append(make_instruction_prompt(instruction, out, inp))
        print(f"  Alpaca alt added     : {len(all_prompts) - alpaca_before:,}")
    except Exception as e2:
        print(f"  Alpaca alt failed ({e2}) — skipping")

In [ ]:
# ── 5. KenSwQuAD (Swahili QA) ─────────────────────────────────
print("Loading KenSwQuAD...")
kenswquad_before = len(all_prompts)
try:
    kenswquad = load_dataset(
        "kenswquad/kenswquad",
        split=f"train[:{KENSWQUAD_SAMPLES}]"
    )
    print(f"  KenSwQuAD columns    : {kenswquad.column_names}")
    for row in kenswquad:
        question = row.get("question", "")
        answer   = row.get("answer",   row.get("answers", ""))
        context  = row.get("context",  "")
        if isinstance(answer, dict):
            answer = answer.get("text", [""])
            answer = answer[0] if answer else ""
        if isinstance(answer, list):
            answer = answer[0] if answer else ""
        if question and answer:
            all_prompts.append(make_instruction_prompt(question, str(answer), context))
    print(f"  KenSwQuAD added      : {len(all_prompts) - kenswquad_before:,}")
except Exception as e:
    print(f"  KenSwQuAD failed ({e}) — skipping")

print(f"\nTotal prompts collected: {len(all_prompts):,}")

In [ ]:
import random

# Shuffle and filter
random.seed(42)
random.shuffle(all_prompts)

# Filter: remove too short or too long
filtered = []
for p in all_prompts:
    token_len = len(tokenizer.encode(p, add_special_tokens=False))
    if 20 < token_len <= MAX_SEQ_LEN:
        filtered.append(p)

print(f"After filtering        : {len(filtered):,} prompts")
print(f"Dropped (too short/long): {len(all_prompts) - len(filtered):,}")
print(f"Estimated steps        : {len(filtered) // (BATCH_SIZE * GRAD_ACCUM):,}")
print(f"Estimated time         : {len(filtered) // (BATCH_SIZE * GRAD_ACCUM) * 35 / 3600:.1f} hours")
print()
print("Sample prompt:")
print(filtered[0])

## 8. Tokenize

In [ ]:
# Build HuggingFace dataset from prompt list
raw_dataset = Dataset.from_dict({"text": filtered})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )

print("Tokenising...")
tokenised = raw_dataset.map(
    tokenize,
    batched=True,
    batch_size=500,
    remove_columns=["text"],
    desc="Tokenising",
)

split = tokenised.train_test_split(test_size=0.02, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]  # unused but kept for Trainer signature

print(f"Train rows             : {len(train_dataset):,}")
print(f"Eval rows              : {len(eval_dataset):,} (unused)")

## 9. Load base model + CPT adapter

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading base model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"" : 0},
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

# Load Stage 1 CPT adapter on top if available
if CPT_ADAPTER_PATH:
    print(f"Loading CPT adapter from: {CPT_ADAPTER_PATH}")
    model = PeftModel.from_pretrained(
        model,
        CPT_ADAPTER_PATH,
        is_trainable=False,   # freeze CPT adapter
    )
    model = model.merge_and_unload()  # merge CPT weights into base
    print("CPT adapter merged into base model.")
else:
    print("No CPT adapter — training from base model.")

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after model load  : {allocated:.2f} GB")

## 10. Apply fresh SFT LoRA adapters

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,  # attention only — MLP aligned in CPT
    bias="none",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params       : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Total params           : {total:,}")

## 11. Training configuration

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Training steps         : {steps:,}")
print(f"Estimated time         : {steps * 35 / 3600:.1f} hours")
print(f"Checkpoints at steps   : {SAVE_STEPS}, {SAVE_STEPS*2}, {SAVE_STEPS*3}")

## 12. Train

In [ ]:
from transformers import Trainer
import time

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print("Starting SFT training...")
print(f"Total steps            : {steps:,}")
print(f"Estimated time         : ~{steps * 35 / 3600:.1f} hours")
print(f"First checkpoint at    : step {SAVE_STEPS} (~{SAVE_STEPS * 35 / 3600:.1f}h)")
print()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/3600:.2f} hours.")
print(f"Final train loss       : {train_result.training_loss:.4f}")

## 13. Save adapter and metrics

In [ ]:
import json

adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

metrics = {
    "stage"            : 2,
    "phase"            : "SFT",
    "base_model"       : MODEL_ID,
    "cpt_adapter"      : CPT_ADAPTER_PATH,
    "train_loss"       : round(train_result.training_loss, 4),
    "train_samples"    : len(train_dataset),
    "total_prompts"    : len(filtered),
    "sawa_samples"     : SAWA_SAMPLES,
    "salt_samples"     : SALT_SAMPLES,
    "aya_samples"      : AYA_SAMPLES,
    "alpaca_samples"   : ALPACA_SAMPLES,
    "kenswquad_samples": KENSWQUAD_SAMPLES,
    "learning_rate"    : LEARNING_RATE,
    "max_seq_len"      : MAX_SEQ_LEN,
    "elapsed_hours"    : round(elapsed/3600, 2),
}

with open(f"{OUTPUT_DIR}/stage2_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Adapter saved          : {adapter_path}")
print(f"Metrics saved          : {OUTPUT_DIR}/stage2_metrics.json")

## 14. Post-SFT smoke test

In [ ]:
model.eval()

GEN_CONFIG = {
    "temperature"        : 0.3,
    "top_p"              : 0.95,
    "top_k"              : 64,
    "max_new_tokens"     : 128,
    "repetition_penalty" : 1.1,
    "do_sample"          : True,
}

def translate(english: str) -> str:
    prompt  = (
        f"<start_of_turn>user\n"
        f"Translate the following English sentence to Swahili.\n"
        f"English: {english}\n"
        f"Swahili:<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            **GEN_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

TEST_PAIRS = [
    ("The students are learning at school today.",
     "Wanafunzi wanasoma shuleni leo."),
    ("She went to the market to buy vegetables and fruit.",
     "Alienda sokoni kununua mboga na matunda."),
    ("The government announced new policies for healthcare.",
     "Serikali ilitangaza sera mpya za huduma za afya."),
    ("Climate change is affecting agriculture across Africa.",
     "Mabadiliko ya tabianchi yanaathiri kilimo katika Afrika yote."),
    ("The beautiful chairs fell on the ground.",
     "Viti vizuri vilianguka chini."),  # noun-class agreement test
]

print("POST-SFT SMOKE TEST")
print("=" * 65)
for en, ref in TEST_PAIRS:
    hyp = translate(en)
    print(f"EN : {en}")
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

## 15. Mini FLORES-200 BLEU check (100 sentences)

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
import time

print("Loading FLORES-200 for mini eval (100 sentences)...")
flores_en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
flores_sw = load_dataset("openlanguagedata/flores_plus", "swh_Latn", split="devtest")

MINI_EVAL = 100
en_sents = flores_en["text"][:MINI_EVAL]
sw_refs  = flores_sw["text"][:MINI_EVAL]

hypotheses = []
start = time.time()

for i, en in enumerate(en_sents):
    hyp = translate(en)
    hypotheses.append(hyp)
    if (i+1) % 25 == 0:
        elapsed = time.time() - start
        print(f"  [{i+1}/{MINI_EVAL}] {elapsed:.0f}s elapsed")

bleu = BLEU(effective_order=True).corpus_score(hypotheses, [sw_refs]).score
chrf = CHRF(word_order=2).corpus_score(hypotheses, [sw_refs]).score

print("\n" + "=" * 55)
print("MINI FLORES-200 RESULTS (100 sentences)")
print("=" * 55)
print(f"BLEU                   : {bleu:.2f}  (target > 27.6)")
print(f"chrF++                 : {chrf:.2f}  (target > 56.8)")
print(f"Stage 0 base model     : ~0 BLEU (was generating Finnish)")
print(f"Swahili Gemma 1B target: 27.6 BLEU / 56.8 chrF++")

# Update and save final metrics
metrics.update({
    "mini_flores_bleu"  : round(bleu, 4),
    "mini_flores_chrf"  : round(chrf, 4),
    "mini_eval_size"    : MINI_EVAL,
})
with open(f"{OUTPUT_DIR}/stage2_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nFinal metrics saved to : {OUTPUT_DIR}/stage2_metrics.json")
print("\nStage 2 SFT complete. Download stage2_sft_adapter/ from output panel.")